# Notebook 05 – Final Forecasts and Stock Decisions

**Purpose.** Consume the winning-model file from **Notebook 04** and:

1. Fit each winning model once on the full series.
2. Produce the **final next-period forecasts** for blood (2025), each medicine (next 12 months), and the weekly/daily pharma groups.
3. Convert forecasts into **stock decisions** — safety stock, reorder point, and suggested order quantity.
4. Export files for the Power BI dashboard (Notebook 06).

**What this notebook does *not* do.** It does not compare models. That comparison lives in Notebook 04 and is treated as fixed here.

**Stock policy**

- Safety stock is based on **weekly** demand variation (not monthly), so that lead-time and reorder decisions use variation on the timescale at which stock is actually managed.
- `safety_stock = Z · σ_weekly · √(lead_time_weeks)`
- `reorder_point = mean_weekly · lead_time_weeks + safety_stock`
- `suggested_order_quantity = forecast_weekly_demand · review_period_weeks + safety_stock`
- `Z = 1.65` (≈95% service level) — configurable.
- These are *planning* outputs; adjust for actual on-hand stock and supplier constraints before ordering.

In [14]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore", message=".*No frequency information.*")
warnings.filterwarnings("ignore", message=".*Maximum Likelihood optimization failed.*")
warnings.filterwarnings("ignore", message=".*Non-invertible starting.*")

PROJECT_DIR = Path(r"C:\Users\Administrator\MediShelf")
PROCESSED   = PROJECT_DIR / "data" / "processed"
PHARMA_PATH = PROCESSED / "pharma_sales_clean.csv"
BLOOD_PATH  = PROCESSED / "blood_demand_clean.csv"
WINNERS_PATH = PROCESSED / "winning_models.csv"

for p in [PHARMA_PATH, BLOOD_PATH, WINNERS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required file missing: {p}")

print("Inputs OK")

Inputs OK


## 1. Load cleaned data and winner list

In [15]:
pharma = pd.read_csv(PHARMA_PATH)
pharma.columns = pharma.columns.str.strip()
if "datum" in pharma.columns:
    pharma = pharma.rename(columns={"datum": "Date"})
pharma["Date"] = pd.to_datetime(pharma["Date"])
pharma = pharma.sort_values("Date").set_index("Date")

blood = pd.read_csv(BLOOD_PATH)
blood["Date"] = pd.to_datetime(blood["Date"])
blood = blood.sort_values("Date").set_index("Date")

winners = pd.read_csv(WINNERS_PATH)
print("Winning models to apply:")
display(winners[["Group", "Design", "Model", "Avg_MASE"]].round(4))

Winning models to apply:


,Group,Design,Model,Avg_MASE
0,BLOOD,Weekly,Training mean,0.7178
1,BLOOD,Weekly,SES,0.7219
2,BLOOD,Weekly,"ARIMA(0,1,1)",0.7274
3,M01AB,Monthly,SES,0.4599
4,M01AB,Monthly,"ARIMA(0,1,1)",0.4608
5,M01AE,Monthly,Training mean,0.8407
6,M01AE,Monthly,SES,0.8464
7,M01AE,Monthly,"ARIMA(0,1,1)",0.8476
8,N02BA,Monthly,"ARIMA(0,1,1)",0.9064
9,N02BE,Monthly,Naive,0.8991


In [16]:
ATC = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

monthly_series = {
    g: pharma[g].resample("MS").mean().asfreq("MS").dropna().astype(float)
    for g in ATC
}

WEEKLY_PHARMA = ["N05C", "R03", "R06"]
weekly_pharma = {
    g: pharma[g].resample("W-SUN").sum().asfreq("W-SUN").dropna().astype(float)
    for g in WEEKLY_PHARMA
}

first_monday = pharma.index.min() + pd.Timedelta(days=(7 - pharma.index.min().weekday()) % 7)
daily_n05c = pharma.loc[first_monday:, "N05C"].asfreq("D").dropna().astype(float)

blood_weekly = blood["HistoricalBloodUsage"].resample("W-SUN").sum().asfreq("W-SUN").dropna().astype(float)
blood_exog = pd.DataFrame({
    "Admissions": blood["HospitalAdmissions"].resample("W-SUN").mean().asfreq("W-SUN"),
    "Events":     blood["Events"].resample("W-SUN").sum().asfreq("W-SUN"),
    "Donors":     blood["BloodDonorsAvailable"].resample("W-SUN").mean().asfreq("W-SUN"),
    "Temp":       blood["Temperature"].resample("W-SUN").mean().asfreq("W-SUN"),
}).dropna().reindex(blood_weekly.index)

print("Series ready")

Series ready


## 2. Final fitting

A dispatch function is applied to each group **using only the winning model(s)** from 04. No competing model is re-fitted here.

In [17]:
def forecast_with_model(series, model_name, horizon, exog_train=None, exog_future=None):
    series = pd.Series(series).dropna().astype(float)

    if model_name == "Training mean":
        return np.repeat(float(series.mean()), horizon)

    if model_name == "Naive":
        return np.repeat(float(series.iloc[-1]), horizon)

    if model_name == "Seasonal naive":
        s = int(min(52, len(series)))
        return np.repeat(float(series.iloc[-s]), horizon)

    if model_name == "SES":
        fit = SimpleExpSmoothing(series, initialization_method="estimated").fit(optimized=True)
        return np.asarray(fit.forecast(horizon), dtype=float)

    if model_name == "ARIMA(0,1,1)":
        fit = ARIMA(series, order=(0,1,1)).fit()
        return np.asarray(fit.forecast(horizon), dtype=float)

    if model_name == "ARIMAX(all exog)":
        assert exog_train is not None and exog_future is not None
        fit = SARIMAX(
            endog=series.values, exog=exog_train.values,
            order=(1,0,1), trend="c",
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False)
        return np.asarray(fit.forecast(horizon, exog=exog_future.values), dtype=float)

    raise ValueError(f"Unsupported model: {model_name}")

### 2.1 Blood — weekly forecast for 2025

Horizon = 52 weeks. If `ARIMAX(all exog)` was the winner in 04, the future exog block is required.

Because we cannot know future exog, and in the synthetic data they are strongly autocorrelated, we extrapolate each exog as its trailing 13-week mean. **This is stated explicitly**: if ARIMAX wins, 05 will show that forecast assumptions are being made about future admissions/events/donors/temperature.

In [18]:
blood_winners = winners[winners.Group == "BLOOD"].sort_values("Avg_MASE")
print("Blood winners:")
display(blood_winners[["Model", "Avg_MASE"]])

HORIZON_BLOOD = 52
future_dates_blood = pd.date_range(
    start=blood_weekly.index[-1] + pd.Timedelta(days=7), periods=HORIZON_BLOOD, freq="W-SUN"
)

blood_forecast_rows = []
for _, row in blood_winners.iterrows():
    model_name = row.Model
    if model_name == "ARIMAX(all exog)":
        exog_train = blood_exog.loc[blood_weekly.index]
        exog_future = pd.DataFrame(
            {col: [exog_train[col].tail(13).mean()] * HORIZON_BLOOD for col in exog_train.columns},
            index=future_dates_blood,
        )
        fc = forecast_with_model(blood_weekly, model_name, HORIZON_BLOOD,
                                 exog_train=exog_train, exog_future=exog_future)
    else:
        fc = forecast_with_model(blood_weekly, model_name, HORIZON_BLOOD)
        exog_future = None

    for d, v in zip(future_dates_blood, fc):
        blood_forecast_rows.append({
            "Item": "BLOOD",
            "Design": "Weekly",
            "Model": model_name,
            "ForecastDate": d,
            "ForecastValue": max(0.0, float(v)),
            "MASE": row.Avg_MASE,
        })

blood_forecast = pd.DataFrame(blood_forecast_rows)
print(f"Blood forecast rows: {len(blood_forecast)}")
display(blood_forecast.head())

Blood winners:


,Model,Avg_MASE
0,Training mean,0.717788
1,SES,0.721893
2,"ARIMA(0,1,1)",0.727367


Blood forecast rows: 156


,Item,Design,Model,ForecastDate,ForecastValue,MASE
0,BLOOD,Weekly,Training mean,2025-01-12,1238.820611,0.717788
1,BLOOD,Weekly,Training mean,2025-01-19,1238.820611,0.717788
2,BLOOD,Weekly,Training mean,2025-01-26,1238.820611,0.717788
3,BLOOD,Weekly,Training mean,2025-02-02,1238.820611,0.717788
4,BLOOD,Weekly,Training mean,2025-02-09,1238.820611,0.717788


### 2.2 Medicine — next 12 monthly forecasts per winning group

In [19]:
pharma_forecast_rows = []
HORIZON_M = 12

pharma_winners = winners[winners.Design == "Monthly"].copy()
for _, row in pharma_winners.iterrows():
    g, m = row.Group, row.Model
    s = monthly_series[g]
    fc = forecast_with_model(s, m, HORIZON_M)
    future_dates = pd.date_range(
        start=s.index[-1] + pd.offsets.MonthBegin(1), periods=HORIZON_M, freq="MS"
    )
    for d, v in zip(future_dates, fc):
        pharma_forecast_rows.append({
            "Item": g, "Design": "Monthly", "Model": m,
            "ForecastDate": d, "ForecastValue": max(0.0, float(v)),
            "MASE": row.Avg_MASE,
        })

pharma_forecast_monthly = pd.DataFrame(pharma_forecast_rows)
print(f"Monthly pharma forecast rows: {len(pharma_forecast_monthly)}")
display(pharma_forecast_monthly.groupby('Item')['ForecastValue'].agg(['mean', 'min', 'max']).round(3))

Monthly pharma forecast rows: 204


,mean,min,max
Item,,,
M01AB,5.534,5.534,5.534
M01AE,3.826,3.785,3.908
N02BA,2.844,2.844,2.844
N02BE,36.894,36.894,36.894
N05B,10.750,10.750,10.750
N05C,0.681,0.596,0.727
R03,4.563,4.531,4.625
R06,1.257,0.988,1.391


### 2.3 Medicine — weekly 4-week forecasts and daily N05C 28-day forecast

In [20]:
weekly_rows = []
HORIZON_W = 4
for _, row in winners[winners.Design == "Weekly"].iterrows():
    if row.Group == "BLOOD":
        continue
    g, m = row.Group, row.Model
    s = weekly_pharma[g]
    fc = forecast_with_model(s, m, HORIZON_W)
    future_dates = pd.date_range(
        start=s.index[-1] + pd.Timedelta(days=7), periods=HORIZON_W, freq="W-SUN"
    )
    for d, v in zip(future_dates, fc):
        weekly_rows.append({
            "Item": g, "Design": "Weekly", "Model": m,
            "ForecastDate": d, "ForecastValue": max(0.0, float(v)),
            "MASE": row.Avg_MASE,
        })
pharma_forecast_weekly = pd.DataFrame(weekly_rows)

daily_rows = []
HORIZON_D = 28
for _, row in winners[winners.Design == "Daily"].iterrows():
    g, m = row.Group, row.Model
    s = daily_n05c
    fc = forecast_with_model(s, m, HORIZON_D)
    future_dates = pd.date_range(
        start=s.index[-1] + pd.Timedelta(days=1), periods=HORIZON_D, freq="D"
    )
    for d, v in zip(future_dates, fc):
        daily_rows.append({
            "Item": g, "Design": "Daily", "Model": m,
            "ForecastDate": d, "ForecastValue": max(0.0, float(v)),
            "MASE": row.Avg_MASE,
        })
pharma_forecast_daily = pd.DataFrame(daily_rows)

print(f"Weekly pharma forecast rows: {len(pharma_forecast_weekly)}")
print(f"Daily N05C forecast rows  : {len(pharma_forecast_daily)}")

Weekly pharma forecast rows: 12
Daily N05C forecast rows  : 28


### 2.4 Combine all forecasts

In [21]:
all_forecasts = pd.concat(
    [blood_forecast, pharma_forecast_monthly, pharma_forecast_weekly, pharma_forecast_daily],
    ignore_index=True,
)
print("Combined forecast rows:", len(all_forecasts))
display(all_forecasts.groupby(["Item", "Design", "Model"])["ForecastValue"].agg(["size", "mean", "sum"]).round(3))

Combined forecast rows: 400


size      mean        sum
Item  Design  Model                                   
BLOOD Weekly  ARIMA(0,1,1)     52  1216.485  63257.235
              SES              52  1238.821  64418.677
              Training mean    52  1238.821  64418.672
M01AB Monthly ARIMA(0,1,1)     12     5.534     66.412
              SES              12     5.534     66.403
M01AE Monthly ARIMA(0,1,1)     12     3.786     45.434
              SES              12     3.785     45.419
              Training mean    12     3.908     46.895
N02BA Monthly ARIMA(0,1,1)     12     2.844     34.124
N02BE Monthly Naive            12    36.894    442.725
N05B  Monthly Naive            12    10.750    129.000
N05C  Daily   SES              28     0.699     19.568
      Monthly ARIMA(0,1,1)     12     0.727      8.721
              SES              12     0.722      8.659
              Training mean    12     0.596      7.153
      Weekly  SES               4     4.729     18.914
R03   Monthly ARIMA(0,1,1)     12     4.532     54.386
              Naive            12     4.625     55.500
              SES              12     4.531     54.371
      Weekly  SES               4    25.029    100.115
R06   Monthly ARIMA(0,1,1)     12     0.988     11.860
              Naive            12     1.391     16.695
              SES              12     1.391     16.695
      Weekly  SES               4     7.580     30.319

## 3. Stock decisions

Stock decisions are made from **weekly variation** of historical demand. Lead time and review period are expressed in weeks; adjust to reflect the real supply chain.

| Parameter | Blood | Medicine |
|---|---|---|
| Service-level factor `Z` | 1.65 (95%) | 1.65 |
| Lead time `L` (weeks) | 2 | 1 |
| Review period `R` (weeks) | 4 | 4 |

Safety stock and reorder point use σ computed **from weekly totals**, not from monthly means — this is the crucial difference that makes the output operationally meaningful.

In [22]:
Z = 1.65

POLICY = {
    "BLOOD": {"lead_time_weeks": 2, "review_weeks": 4},
    "M01AB": {"lead_time_weeks": 1, "review_weeks": 4},
    "M01AE": {"lead_time_weeks": 1, "review_weeks": 4},
    "N02BA": {"lead_time_weeks": 1, "review_weeks": 4},
    "N02BE": {"lead_time_weeks": 1, "review_weeks": 4},
    "N05B":  {"lead_time_weeks": 1, "review_weeks": 4},
    "N05C":  {"lead_time_weeks": 1, "review_weeks": 4},
    "R03":   {"lead_time_weeks": 1, "review_weeks": 4},
    "R06":   {"lead_time_weeks": 1, "review_weeks": 4},
}

# Weekly history per item (used to compute σ weekly):
weekly_history = {"BLOOD": blood_weekly}
for g in ATC:
    weekly_history[g] = pharma[g].resample("W-SUN").sum().asfreq("W-SUN").dropna().astype(float)

stock_rows = []
for item, s in weekly_history.items():
    policy = POLICY[item]
    mean_weekly = float(s.mean())
    sigma_weekly = float(s.std())

    ss = Z * sigma_weekly * np.sqrt(policy["lead_time_weeks"])
    rop = mean_weekly * policy["lead_time_weeks"] + ss

    # Use the item's own forecast (weekly model where available; else
    # derive weekly-equivalent from the monthly model)
    weekly_forecast = all_forecasts[(all_forecasts.Item == item) & (all_forecasts.Design == "Weekly")]
    if not weekly_forecast.empty:
        avg_weekly_forecast = float(weekly_forecast.ForecastValue.mean())
    else:
        monthly = all_forecasts[(all_forecasts.Item == item) & (all_forecasts.Design == "Monthly")]
        avg_weekly_forecast = float(monthly.ForecastValue.mean()) / 4.345 if not monthly.empty else mean_weekly

    oq = avg_weekly_forecast * policy["review_weeks"] + ss

    stock_rows.append({
        "Item": item,
        "Mean_Weekly_Demand": round(mean_weekly, 2),
        "Sigma_Weekly_Demand": round(sigma_weekly, 2),
        "Lead_Time_Weeks": policy["lead_time_weeks"],
        "Review_Weeks": policy["review_weeks"],
        "Safety_Stock": round(ss, 2),
        "Reorder_Point": round(rop, 2),
        "Suggested_Order_Quantity": round(oq, 2),
        "Service_Level_Z": Z,
    })

stock_decisions = pd.DataFrame(stock_rows)
print("Stock decisions (based on weekly variation):")
display(stock_decisions)

Stock decisions (based on weekly variation):


,Item,Mean_Weekly_Demand,Sigma_Weekly_Demand,Lead_Time_Weeks,Review_Weeks,Safety_Stock,Reorder_Point,Suggested_Order_Quantity,Service_Level_Z
0,BLOOD,1238.82,195.08,2,4,455.21,2932.85,5380.71,1.65
1,M01AB,35.10,8.62,1,4,14.22,49.32,19.31,1.65
2,M01AE,27.17,7.04,1,4,11.62,38.79,15.14,1.65
3,N02BA,27.06,8.09,1,4,13.34,40.40,15.96,1.65
4,N02BE,208.63,76.07,1,4,125.51,334.14,159.48,1.65
5,N05B,61.74,22.44,1,4,37.02,98.76,46.92,1.65
6,N05C,4.14,3.13,1,4,5.16,9.30,24.08,1.65
7,R03,38.44,22.90,1,4,37.79,76.23,137.90,1.65
8,R06,20.22,11.38,1,4,18.78,39.00,49.10,1.65


### 3.1 How to read the table

- **Safety stock** — buffer of units held to absorb normal weekly variation during the lead time.
- **Reorder point** — when on-hand stock falls to this level, place a new order.
- **Suggested order quantity** — units to order at each review to cover forecast demand for the review period plus safety stock.
- These are operational *planning* values. In real life, adjust for shelf life, MOQs, on-hand stock, and supplier constraints.

## 4. Export for the Power BI dashboard

Two files are written to `data/processed/`:

- `final_forecasts.csv` — long-format forecast table.
- `stock_decisions.csv` — one row per item with policy outputs.

Together with `pharma_sales_clean.csv` and `blood_demand_clean.csv` these feed the dashboard built in Notebook 06.

In [23]:
forecast_path = PROCESSED / "final_forecasts.csv"
stock_path    = PROCESSED / "stock_decisions.csv"

# Save forecasts
export_forecasts = all_forecasts.copy()
export_forecasts["ForecastDate"] = export_forecasts["ForecastDate"].dt.strftime("%Y-%m-%d")
export_forecasts.to_csv(forecast_path, index=False)

# Save stock decisions
stock_decisions.to_csv(stock_path, index=False)

print("Wrote:", forecast_path, f"({len(export_forecasts)} rows)")
print("Wrote:", stock_path, f"({len(stock_decisions)} rows)")

Wrote: C:\Users\Administrator\MediShelf\data\processed\final_forecasts.csv (400 rows)
Wrote: C:\Users\Administrator\MediShelf\data\processed\stock_decisions.csv (9 rows)


In [24]:
# Quick sanity check: reload and confirm the files saved correctly
fc = pd.read_csv(forecast_path)
sd = pd.read_csv(stock_path)

assert len(fc) > 0 and len(sd) > 0, "Files are empty!"
assert set(sd.Item) == set(weekly_history.keys()), "Mismatch in tracked items"
assert (sd.Safety_Stock >= 0).all(), "Found negative safety stock"

print("Reload OK.")
print("Forecast rows:", len(fc))
print("Decision rows:", len(sd))

Reload OK.
Forecast rows: 400
Decision rows: 9
